# F1 Championship Forecast — canonical exploration

This notebook uses the same processed schema, predeclared model, and prediction path as the release pipeline.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join('..', 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from model import FEATURE_COLUMNS, train_model
from predict import predict_championship

PROC = os.path.join('..', 'data', 'processed')
sns.set_theme(style='darkgrid')

## Processed features

In [ ]:
features = pd.read_csv(os.path.join(PROC, 'features.csv'))
print(f'Features shape: {features.shape}')
print(f"Missing championship targets: {features['champ_position'].isna().sum()}")

cols_of_interest = [
    'prev_season_points_sum', 'prev_season_sprint_points_sum',
    'prev_season_avg_finish_pos', 'prev_season_win_rate',
    'prev_season_dnf_rate', 'prev_team_final_points',
    'prev_team_final_position', 'champ_position'
]
sns.heatmap(features[cols_of_interest].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Canonical feature correlation matrix')
plt.tight_layout()

## Train the predeclared forecast model and inspect 2023

In [ ]:
reg_model, clf_model = train_model(forecast_year=2023, skip_rolling_evaluation=True)
predictions = predict_championship(2023)
predictions.head()

In [ ]:
valid = predictions.dropna(subset=['Actual Position'])
plt.figure(figsize=(8, 8))
plt.scatter(valid['Actual Position'], valid['Predicted Position'], color='steelblue', s=80)
lims = [1, int(valid['Actual Position'].max()) + 1]
plt.plot(lims, lims, 'r--', alpha=0.5, label='Perfect prediction')
plt.xlabel('Actual Championship Position')
plt.ylabel('Predicted Position (lower = better)')
plt.title('2023 F1 Championship — predicted versus actual')
plt.legend()
plt.tight_layout()

## Operational feature importance

In [ ]:
importance = pd.Series(reg_model.feature_importances_, index=FEATURE_COLUMNS).sort_values()
importance.plot(kind='barh', title='Feature importance — Random Forest', color='steelblue')
plt.xlabel('Importance')
plt.tight_layout()